In [ ]:
"""SCI风格热图（最终投稿版）
优化内容：
1. 主热图仅保留整体外边框
2. 行平均值仅保留整体外边框
3. 列平均值仅保留整体外边框
4. 内部无任何边框线
5. 边框宽度统一为1磅
6. 行列平均值与热图共用同一配色
7. SCI红-粉-蓝渐变配色
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle

warnings.filterwarnings('ignore')

# ========================== 参数区 ==========================

# 文件路径
result_dir = r"H:\图像标记\上下采样"

# 方法与模型
preprocess_methods = [
    'RAW', 'MA', 'SG', 'WD', 'D1st',
    'D2nd', 'SNV', 'MSC', 'Deconv', 'MinMax'
]

models = ['SVC', 'RF', 'XGBoost', 'PLS-DA', 'KNN']

# 字号
FONT_CELL_VALUE = 26
FONT_AXIS_LABEL = 16
FONT_TITLE = 17
FONT_CBAR = 11
FONT_TICK = 15

# 画布
FIG_SIZE = (12, 8)
DPI = 300

# 精度范围
VMAX = 1.0
CV_VMIN = 0.6
TEST_VMIN = 0.0

# 间距
GAP = 0.18

# 边框
BORDER_COLOR = '#000000'
BORDER_WIDTH = 1.2

# 字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# ==========================================================
# SCI配色（深 → 浅）
# #D83F65 → #E896AB → #89BED1
# ==========================================================
custom_cmap = plt.get_cmap('YlOrRd')

# ==========================================================
# 读取结果
# ==========================================================
cv_accuracy_matrix = np.zeros((len(preprocess_methods), len(models)))
test_accuracy_matrix = np.zeros((len(preprocess_methods), len(models)))

for pre_idx, pre_name in enumerate(preprocess_methods):

    result_file = os.path.join(
        result_dir,
        f"result{pre_name}.xlsx"
    )

    if not os.path.exists(result_file):

        print(f"缺少文件：{result_file}")
        continue

    df = pd.read_excel(result_file)

    for mod_idx, mod_name in enumerate(models):

        row = df[df['模型'] == mod_name]

        if not row.empty:

            cv_accuracy_matrix[pre_idx, mod_idx] = \
                row['交叉验证精度'].values[0]

            test_accuracy_matrix[pre_idx, mod_idx] = \
                row['平均精度'].values[0]

# ==========================================================
# DataFrame
# ==========================================================
cv_df = pd.DataFrame(
    cv_accuracy_matrix,
    index=preprocess_methods,
    columns=models
)

test_df = pd.DataFrame(
    test_accuracy_matrix,
    index=preprocess_methods,
    columns=models
)

# ==========================================================
# 热图函数
# ==========================================================
def plot_heatmap(data, title, save_path, vmin):

    row_mean = data.mean(axis=1)
    col_mean = data.mean(axis=0)

    n_rows, n_cols = data.shape

    fig, ax = plt.subplots(
        figsize=FIG_SIZE,
        dpi=DPI
    )

    # ======================================================
    # 主热图
    # ======================================================
    im = ax.imshow(
        data.values,
        cmap=custom_cmap,
        aspect='auto',
        vmin=vmin,
        vmax=VMAX
    )

    # ======================================================
    # 主热图数值
    # ======================================================
    for i in range(n_rows):

        for j in range(n_cols):

            val = data.iloc[i, j]

            # 中间值黑色，两端白色
            norm_val = (val - vmin) / (VMAX - vmin)
            
            if norm_val <= 0.65:
                color = 'black'
            else:
                color = 'white'

            ax.text(
                j,
                i,
                f"{val:.4f}",
                ha='center',
                va='center',
                fontsize=FONT_CELL_VALUE,
                fontweight='bold',
                fontname='Times New Roman',
                color=color
            )

    # ======================================================
    # 行平均值区域（使用同一配色）
    # ======================================================
    row_x = n_cols + GAP

    norm_row = plt.Normalize(vmin=vmin, vmax=VMAX)

    for i in range(n_rows):

        val = row_mean.iloc[i]

        rect = Rectangle(
            (row_x - 0.5, i - 0.5),
            1,
            1,
            facecolor=custom_cmap(norm_row(val)),
            edgecolor='none'
        )

        ax.add_patch(rect)

       # 中间值黑色，两端白色
        norm_val = (val - vmin) / (VMAX - vmin)
        
        if norm_val <= 0.65:
            color = 'black'
        else:
            color = 'white'

        ax.text(
            row_x,
            i,
            f"{val:.4f}",
            ha='center',
            va='center',
            fontsize=FONT_CELL_VALUE,
            fontweight='bold',
            fontname='Times New Roman',
            color=color
        )

    # ======================================================
    # 列平均值区域（使用同一配色）
    # ======================================================
    col_y = n_rows + GAP

    norm_col = plt.Normalize(vmin=vmin, vmax=VMAX)

    for j in range(n_cols):

        val = col_mean.iloc[j]

        rect = Rectangle(
            (j - 0.5, col_y - 0.5),
            1,
            1,
            facecolor=custom_cmap(norm_col(val)),
            edgecolor='none'
        )

        ax.add_patch(rect)

        # 中间值黑色，两端白色
        norm_val = (val - vmin) / (VMAX - vmin)
        
        if norm_val <= 0.65:
            color = 'black'
        else:
            color = 'white'

        ax.text(
            j,
            col_y,
            f"{val:.4f}",
            ha='center',
            va='center',
            fontsize=FONT_CELL_VALUE,
            fontweight='bold',
            fontname='Times New Roman',
            color=color
        )

    # ======================================================
    # 主热图整体外边框
    # ======================================================
    main_border = Rectangle(
        (-0.5, -0.5),
        n_cols,
        n_rows,
        fill=False,
        edgecolor=BORDER_COLOR,
        linewidth=BORDER_WIDTH
    )

    ax.add_patch(main_border)

    # ======================================================
    # 行平均值整体外边框
    # ======================================================
    row_mean_border = Rectangle(
        (row_x - 0.5, -0.5),
        1,
        n_rows,
        fill=False,
        edgecolor=BORDER_COLOR,
        linewidth=BORDER_WIDTH
    )

    ax.add_patch(row_mean_border)

    # ======================================================
    # 列平均值整体外边框
    # ======================================================
    col_mean_border = Rectangle(
        (-0.5, col_y - 0.5),
        n_cols,
        1,
        fill=False,
        edgecolor=BORDER_COLOR,
        linewidth=BORDER_WIDTH
    )

    ax.add_patch(col_mean_border)

    # ======================================================
    # X轴放顶部
    # ======================================================
    ax.xaxis.tick_top()

    # ======================================================
    # 坐标轴标签
    # ======================================================
    ax.set_xticks(np.arange(n_cols))
    ax.set_yticks(np.arange(n_rows))

    ax.set_xticklabels(
        models,
        fontsize=FONT_AXIS_LABEL,
        fontweight='bold'
    )

    ax.set_yticklabels(
        preprocess_methods,
        fontsize=FONT_AXIS_LABEL,
        fontweight='bold'
    )

    plt.setp(
        ax.get_xticklabels(),
        rotation=45,
        ha='left',
        rotation_mode='anchor'
    )

    # ======================================================
    # Mean标签
    # ======================================================
    ax.text(
        row_x,
        -1.0,
        'Row Mean',
        ha='center',
        va='center',
        fontsize=FONT_AXIS_LABEL,
        fontname='Times New Roman',
        fontweight='bold'
    )

    ax.text(
        -1.2,
        col_y,
        'Col Mean',
        ha='center',
        va='center',
        fontsize=FONT_AXIS_LABEL,
        fontname='Times New Roman',
        fontweight='bold'
    )

    # ======================================================
    # 坐标范围
    # ======================================================
    ax.set_xlim(-0.5, row_x + 0.5)
    ax.set_ylim(col_y + 0.5, -0.5)

    # ======================================================
    # 去掉默认spines
    # ======================================================
    for spine in ax.spines.values():
        spine.set_visible(False)

    # ======================================================
    # Colorbar
    # ======================================================
    cbar = plt.colorbar(im, ax=ax)

    cbar.ax.tick_params(labelsize=FONT_TICK)

    cbar.outline.set_linewidth(1)

    # ======================================================
    # 标题
    # ======================================================
    ax.set_title(
        title,
        fontsize=FONT_TITLE,
        fontweight='bold',
        pad=40
    )

    plt.tight_layout()

    # ======================================================
    # 保存
    # ======================================================
    plt.savefig(
        save_path,
        dpi=600,
        bbox_inches='tight'
    )

    plt.show()

# ==========================================================
# 输出热图
# ==========================================================
cv_path = os.path.join(
    result_dir,
    "SCI_交叉验证热图8.png"
)

test_path = os.path.join(
    result_dir,
    "SCI_测试精度热图8.png"
)

plot_heatmap(
    cv_df,
    '不同预处理 + 模型交叉验证精度热图',
    cv_path,
    CV_VMIN
)

plot_heatmap(
    test_df,
    '不同预处理 + 模型平均测试精度热图',
    test_path,
    TEST_VMIN
)

# ==========================================================
# 完成
# ==========================================================
print("\n✅ SCI热图生成完成！")
print(f"📊 交叉验证热图：{cv_path}")
print(f"📊 测试精度热图：{test_path}")